# 2. Indexing - Chunking, Embeddings & ChromaDB

This notebook takes the PubMed documents collected in Notebook 1 and indexes them into a ChromaDB vector store using LlamaIndex. The pipeline performs sentence-based chunking, generates embeddings with a HuggingFace model (BGE), and persists everything in ChromaDB for downstream retrieval and generation.

In [1]:
import json
from pathlib import Path

import chromadb
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings

In [2]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
CHROMA_DIR = BASE_DIR / "chroma_db"
CHROMA_DIR.mkdir(exist_ok=True)

## 2.1 Load documents

In [3]:
with open(DATA_DIR / "pubmed_documents.json", "r") as f:
    raw_docs = json.load(f)
print(f"Loaded {len(raw_docs):,} documents")

Loaded 21,903 documents


In [4]:
documents = []
for doc_id, text in raw_docs.items():
    doc = Document(
        text=text,
        metadata={"source_id": doc_id}
    )
    documents.append(doc)

print(f"Created {len(documents)} LlamaIndex Document objects")
print(f"Example metadata: {documents[0].metadata}")
print(f"Example text (first 200 chars): {documents[0].text[:200]}")

Created 21903 LlamaIndex Document objects
Example metadata: {'source_id': 'Document_1053429'}
Example text (first 200 chars): title: Sham peer review
content: peer review could draw out the process by legal maneuvering, and the fairness of a peer review that has been unduly delayed has been called into question. Many medical


## 2.2 Configure embedding model

In [5]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-base-en-v1.5"
)

# Test embedding
test_embedding = embed_model.get_text_embedding("test query")
print(f"Embedding dimension: {len(test_embedding)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 768


## 2.3 Configure chunking

In [6]:
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

Settings.embed_model = embed_model
Settings.text_splitter = splitter

# Preview chunking on one document
test_nodes = splitter.get_nodes_from_documents([documents[0]])
print(f"Document 0 split into {len(test_nodes)} chunks")
for i, node in enumerate(test_nodes[:3]):
    print(f"  Chunk {i}: {len(node.text)} chars, metadata: {node.metadata}")

Document 0 split into 1 chunks
  Chunk 0: 628 chars, metadata: {'source_id': 'Document_1053429'}


## 2.4 Create ChromaDB index

In [7]:
# Initialize ChromaDB persistent client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Create or get collection
chroma_collection = chroma_client.get_or_create_collection(
    name="pubmed_rag",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB collection 'pubmed_rag' - existing docs: {chroma_collection.count()}")

ChromaDB collection 'pubmed_rag' - existing docs: 21903


In [8]:
# Setup vector store
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build index - this chunks documents, generates embeddings, and stores in ChromaDB
print("Building index (chunking + embedding + storing)...")
print("This may take several minutes depending on your hardware...")

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True
)

print(f"\nIndexing complete!")
print(f"Total chunks in ChromaDB: {chroma_collection.count()}")

Building index (chunking + embedding + storing)...
This may take several minutes depending on your hardware...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

...

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1423 [00:00<?, ?it/s]


Indexing complete!
Total chunks in ChromaDB: 43806


## 2.5 Sanity check

In [9]:
# Quick test - retrieval only (no LLM needed)
test_query = "Is anorectal endosonography valuable in dyschesia?"
print(f"Test query: {test_query}\n")

retriever = index.as_retriever(similarity_top_k=5)
results = retriever.retrieve(test_query)

print(f"Retrieved {len(results)} chunks:")
for i, node in enumerate(results):
    print(f"\n--- Result {i+1} (score: {node.score:.4f}) ---")
    print(f"Source: {node.metadata.get('source_id', 'N/A')}")
    print(f"Text: {node.text[:200]}...")

Test query: Is anorectal endosonography valuable in dyschesia?



Retrieved 3 chunks:

--- Result 1 (score: 0.8247) ---
Source: Document_47183
Text: Dyschesia can be provoked by inappropriate defecation movements. The aim of this prospective study was to demonstrate dysfunction of the anal sphincter and/or the musculus (m.) puborectalis in patient...

--- Result 2 (score: 0.7348) ---
Source: Document_627762
Text: title: Dyssynergia
content: a decrease in intrarectal pressure defecation can occur. Anal sphincter dyssynergia tends to be one of the most predicted diagnoses with a patient suffering from symptoms l...

--- Result 3 (score: 0.7344) ---
Source: Document_216720
Text: title: Dyssynergia
content: doctors. An abnormal or prolonged time of expulsion of the balloon is seen as a problem in the anorectum region of the body and may lead to the diagnosis of dyssynergia, si...


In [10]:
print(f"Final ChromaDB stats:")
print(f"  Collection: pubmed_rag")
print(f"  Total chunks: {chroma_collection.count()}")
print(f"  Original documents: {len(raw_docs)}")
print(f"  Avg chunks per document: {chroma_collection.count() / len(raw_docs):.1f}")

Final ChromaDB stats:
  Collection: pubmed_rag
  Total chunks: 43806
  Original documents: 21903
  Avg chunks per document: 2.0


---

Index is persisted in the `chroma_db/` directory. The next notebook will load this index for retrieval and generation.